In [1]:
import xml.etree.ElementTree as ET
import re
import os
import spacy
import scispacy
from datetime import datetime

### XML Structure
- teiHeader
    - fileDesc
        - titleStmt
            - title
        - publicationStmt
            - date
        - sourceDesc
            - biblStruct
                - idno
    - encodingDesc
    - profileDesc

### Parsed (XML-to-RDF) Data Structure
- [
    - [
        - section_number,
        - section_title,
        - {
            - paragraph_number: paragraph_text,
        - }
    - ],
- ]

# RDF Generator

**Helper functions:** \
*`_parse_document_xml()`*, *`_get_direct_subsections()`* => `generate_document_rdf()`

In [2]:
def _parse_document_xml(file: str) -> list:

    # LOADING XML AND CREATING ROOT
    xml_file = f'{file}'
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # EXTRACTING METADATA (TITLE, PUBLICATION_DATE and DOI)
    metadata = []

    for elem in root:
        if elem.tag[29:] == 'teiHeader':
            for sub_elem1 in elem:
                if sub_elem1.tag[29:] == 'fileDesc':
                    publication_info = {}
                    for sub_elem2 in sub_elem1:
                        if sub_elem2.tag[29:] == 'titleStmt':
                            for sub_elem3 in sub_elem2:
                                if sub_elem3.tag[29:] == 'title':
                                    publication_info['Title'] = sub_elem3.text
                        if sub_elem2.tag[29:] == 'publicationStmt':
                            for sub_elem3 in sub_elem2:
                                if sub_elem3.tag[29:] == 'date':
                                    publication_info['Publication Date'] = sub_elem3.attrib['when']
                        if sub_elem2.tag[29:] == 'sourceDesc':
                            for sub_elem3 in sub_elem2:
                                if sub_elem3.tag[29:] == 'biblStruct':
                                    for sub_elem4 in sub_elem3:
                                        if sub_elem4.tag[29:] == 'idno':
                                            publication_info['DOI'] = sub_elem4.text
                    metadata.append(['0', 'Metadata', publication_info])
    
    # EXTRACTING ABSTRACT AND NUMBERING THE PARAGRAPHS
    abstract = []

    for elem in root:
        if elem.tag[29:] == 'teiHeader':
            for sub_elem1 in elem:
                if sub_elem1.tag[29:] == 'profileDesc':
                    for sub_elem2 in sub_elem1:
                        if sub_elem2.tag[29:] == 'abstract':
                            for sub_elem3 in sub_elem2:
                                if sub_elem3.tag[29:] == 'div':
                                    list_of_paragraphs = {}
                                    paragraph_number = 0
                                    for sub_elem4 in sub_elem3:
                                        if sub_elem4.tag[29:] == 'p':
                                            paragraph_number += 1
                                            list_of_paragraphs[paragraph_number] = ET.tostring(sub_elem4, encoding='unicode')
                                    abstract.append(['0', 'Abstract', list_of_paragraphs])
                                 
    # EXTRACTING SECTIONS
    # need to normalize the section number (line 17)
    # compare Rojas and Wolf section number
    # \ issue in RDF, see Koshkava 2014 paper
    list_of_sections = []

    for elem in root:
        if (elem.tag[29:] == 'text'):
            for sub_elem1 in elem:
                if (sub_elem1.tag[29:] == 'body'):
                    for sub_elem2 in sub_elem1:
                        if sub_elem2.tag[29:] == 'div':
                            section_number = ''
                            section_name = ''
                            list_of_paragraphs = []
                            for sub_elem3 in sub_elem2:
                                if sub_elem3.tag[29:] == 'head':
                                    if bool(sub_elem3.attrib):
                                        section_number = str(sub_elem3.attrib)
                                        if section_number[-3] == '.':
                                            section_number = section_number[7:-3]
                                        else:
                                            section_number = section_number[7:-2]
                                    else:
                                        section_number = 'NO_SECTION_NUMBER'
                                    section_name = sub_elem3.text
                                if sub_elem3.tag[29:] == 'p':
                                    list_of_paragraphs.append(ET.tostring(sub_elem3, encoding='unicode'))
                            # commented logic skips NO_SECTION_NUMBER with no paragraphs, i.e., Table 1, Table 2, .....
                            # if section_number == 'NO_SECTION_NUMBER' and not bool(list_of_paragraphs):
                            if section_number == 'NO_SECTION_NUMBER':
                                pass
                            else:
                                list_of_sections.append([section_number, section_name, list_of_paragraphs])
                                
    # NUMBERING THE PARAGRAPHS OF SECTIONS 
    for section in list_of_sections:
        dict_ = {i + 1: section[2][i] for i in range(len(section[2]))}
        section.append(dict_)
        section.remove(section[2])
        
    # MERGING ABSTRACT WITH OTHER DOCUMENT PARTS
    document = abstract + list_of_sections

    # PREPROCESSING TEXT
    starting_p_tag_pattern = r'<ns0:p[^>]+>'
    ending_p_tag_pattern = '</ns0:p>'
    starting_ref_tag_pattern = r'<ns0:ref[^>]+>'
    ending_ref_tag_pattern = '</ns0:ref>'
    ref_pattern = r'<ref>.*?</ref>'  # temporary for removing ref tag

    for record in document:
        for paragraph_number, paragraph_text in record[2].items():
            text = paragraph_text
            text = re.sub(starting_p_tag_pattern, '', text)
            text = re.sub(ending_p_tag_pattern, '', text)
            text = re.sub(r'<ns0:ref type="(?:table|figure|formula)">([^<]+)</ns0:ref>', r'\1', text)
            text = re.sub(r'<ns0:ref[^>]*type="(?:table|figure|formula)"[^>]*>([^<]+)</ns0:ref>', r'\1', text)
            text = re.sub(starting_ref_tag_pattern, '<ref>', text)
            text = re.sub(ending_ref_tag_pattern, '</ref>', text)
            text = re.sub(ref_pattern, '', text)
            record[2][paragraph_number] = text
    
    # MERGING METADATA WITH OTHER DOCUMENT PARTS
    document = metadata + document
    
    return document


In [1]:
# SUBSECTIONS CHECKER FOR A SECTION
def _get_direct_subsections(document, doi, section_number):
    section_ids = []

    for record in document:
        pattern = rf'^{section_number}\.[^.]+$'
        
        if re.search(pattern, record[0]):
            # section_id = doi + '_' + str(record[0])
            # section_ids.append('data:'+section_id)
            section_id = f'{doi}_S{record[0]}'
            section_ids.append(f'data:{section_id}')

    return ', '.join(section_ids)


### Document RDF Writer

In [4]:
# ============================================================
# RDF WRITER FOR DOCUMENT
# ============================================================

def generate_document_rdf(file: str) -> tuple[str, str]:

    document = _parse_document_xml(file) 

    # use doi for generating unique rdf identifier for the document
    doi = document[0][2]['DOI']

    if not doi:
        raise ValueError('DOI not found!')
    
    document_id = doi.replace('/', '_')

    # --------------------------------------------------
    # process each record in document
    # --------------------------------------------------

    xsd_string = '^^xsd:string'
    xsd_date = '^^xsd:date'
    xsd_non_neg_int = '^^xsd:nonNegativeInteger'
    
    rdf = ''
    
    for record in document:
        
        # process metadata
        if record[1] == 'Metadata':

            # document parts directly contained by the document
            abstract_id = f'data:{document_id}_A'
            section_ids = []

            for section_number in document:
                if section_number[0] != '0' and '.' not in section_number[0]:
                    section_id = f'{document_id}_S{section_number[0]}'
                    section_ids.append(f'data:{section_id}')

            directly_contained_sections = ', '.join(section_ids)

            # generate turtle representation
            rdf += f"data:Publication_{document_id} rdf:type onner:ScholarlyPublication ;\n"
            rdf += f"onner:publicationTitle '{record[2]['Title']}'{xsd_string} ;\n"
            rdf += f"onner:publicationDate '{record[2]['Publication Date']}'{xsd_date} ;\n"
            rdf += f"onner:doi '{doi}'{xsd_string} ;\n"
            rdf += f"onner:directlyContainsDocumentPart {abstract_id}, {directly_contained_sections} .\n\n"

        # process abstract and its document parts
        elif record[1] == 'Abstract':

            # get the immediate next section of the abstract 
            next_index = document.index(record) + 1
            next_section = document[next_index][0]
            
            # paragraphs directly contained by the abstract
            paragraph_ids = []

            for paragraph_number, _ in record[2].items():
                paragraph_id = f'{document_id}_A_P{paragraph_number}'
                paragraph_ids.append(f'data:{paragraph_id}')

            paragraph_ids_joined = ', '.join(paragraph_ids)

            # generate turtle representation
            rdf += f"data:{document_id}_A rdf:type onner:Abstract ;\n"
            rdf += f"onner:nextDocumentPart {paragraph_ids[0]} ;\n"    # NEXT DOC PART AFTER ABSTRACT
            rdf += f"onner:directlyContainsDocumentPart {paragraph_ids_joined} .\n\n"

            # generate turtle representation for each paragraph of the abstract
            for paragraph_number, paragraph_text in record[2].items():
                # replacing ' with \' in text
                # if "'" in paragraph_text:
                #     paragraph_text = paragraph_text.replace("'", r"\'")

                paragraph_text = paragraph_text.replace('\\', '\\\\')
                paragraph_text = paragraph_text.replace("'", r"\'")

                rdf += f"data:{document_id}_A_P{paragraph_number} rdf:type onner:Paragraph ;\n"
                rdf += f"onner:positionInParentDocumentPart '{paragraph_number}'{xsd_non_neg_int} ;\n"

                if paragraph_number == len(paragraph_ids):
                    rdf += f"onner:nextDocumentPart data:{document_id}_S{next_section} ;\n"
                else:
                    rdf += f"onner:nextDocumentPart data:{document_id}_A_P{paragraph_number+1} ;\n"

                rdf += f"onner:paragraphText '{paragraph_text}'{xsd_string} .\n\n"

        # process section and its document parts
        else:

            section_number = record[0]
            section_name = record[1]
            next_index = document.index(record) + 1
            paragraph_ids = []

            if next_index < len(document):
                next_section = document[next_index][0]

            # if block executes if no paragraphs exist between a section and its immediate subsection
            # else block executes if paragraphs exist between a section and its immediate subsection
            if not bool(record[2]):
                directly_contained_sections = _get_direct_subsections(document, document_id, section_number)

                rdf += f"data:{document_id}_S{section_number} rdf:type onner:Section ;\n"
                rdf += f"onner:sectionTitle '{section_name}'{xsd_string} ;\n"
                rdf += f"onner:sectionNumber '{section_number}'{xsd_string} ;\n"
                rdf += f"onner:nextDocumentPart data:{document_id}_S{next_section} ;\n"
                rdf += f"onner:directlyContainsDocumentPart {directly_contained_sections} .\n\n"
            
            else:
                directly_contained_sections = _get_direct_subsections(document, document_id, section_number)

                for paragraph_number, _ in record[2].items():
                    # paragraph_id = document_id + '_S' + str(section_number) + '_P' + str(paragraph_number)
                    paragraph_id = f'{document_id}_S{section_number}_P{paragraph_number}'
                    paragraph_ids.append(f'data:{paragraph_id}')

                paragraph_ids_joined = ', '.join(paragraph_ids)

                rdf += f"data:{document_id}_S{section_number} rdf:type onner:Section ;\n"
                rdf += f"onner:sectionTitle '{section_name}'{xsd_string} ;\n"
                rdf += f"onner:sectionNumber '{section_number}'{xsd_string} ;\n"
                rdf += f"onner:nextDocumentPart {paragraph_ids[0]} ;\n"  # NEXT DOC PART AFTER SECTION

                if bool(directly_contained_sections):
                    rdf += f"onner:directlyContainsDocumentPart {paragraph_ids_joined}, {directly_contained_sections} .\n\n"
                else:
                    rdf += f"onner:directlyContainsDocumentPart {paragraph_ids_joined} .\n\n"

                for paragraph_number, paragraph_text in record[2].items():
                    # replacing ' with \' in text
                    if "'" in paragraph_text:
                        paragraph_text = paragraph_text.replace("'", r"\'")
 
                    rdf += f"data:{document_id}_S{section_number}_P{paragraph_number} rdf:type onner:Paragraph ;\n"
                    rdf += f"onner:positionInParentDocumentPart '{paragraph_number}'{xsd_non_neg_int} ;\n"

                    # EndOfDocument appears only after the last paragraph of the last section of the document
                    # a document ends with a paragraph of a section, not right after a section without its paragraphs
                    # e.g., EndOfDocument appears after the last paragraph of the 'conclusion' section
                    if paragraph_number == len(paragraph_ids):
                        if next_index == len(document):
                            rdf += f"onner:nextDocumentPart data:{document_id}_EndOfDocument ;\n"
                        else:
                            rdf += f"onner:nextDocumentPart data:{document_id}_S{next_section} ;\n"
                    else:
                        rdf += f"onner:nextDocumentPart data:{document_id}_S{section_number}_P{paragraph_number+1} ;\n"

                    rdf += f"onner:paragraphText '{paragraph_text}'{xsd_string} .\n\n"

    rdf += f"data:{document_id}_EndOfDocument rdf:type onner:EndOfDocument .\n"

    return document_id, rdf


### Entity RDF Writer

In [5]:
# ============================================================
# RDF WRITER FOR ENTITIES
# ============================================================

def generate_entity_rdf(entities, labeling_schema, model):

    xsd_string = '^^xsd:string'
    xsd_datetime = '^^xsd:dateTime'
    xsd_non_neg_int = '^^xsd:nonNegativeInteger'
    
    rdf = ''
    labels_found = {}

    schema_name = labeling_schema['name']
    schema_labels = labeling_schema['labels']

    # WRITING RDF
    for i in entities:
        paragraph_id = i['paragraph_id']
        entity_list = i['entities']

        if bool(entity_list):    # if no term found in paragraph
            entity_ids = ['data:'+i['entity_id'] for i in entity_list]
            entity_ids_joined = ', '.join(entity_ids) 
            
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm {entity_ids_joined} .\n\n"
            
            for info in entity_list:
                rdf += f"data:{info['entity_id']} rdf:type onner:LabeledTerm ;\n"    # deal with atomic and compound terms
                rdf += f"onner:labeledTermText '{info['entity_text']}'{xsd_string} ;\n"
                rdf += f"onner:offset '{info['offset']}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:length '{info['length']}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:labeledTermDirectlyContainedBy data:{paragraph_id} ;\n"
                rdf += f"onner:hasLabeledTermStatus data:Candidate_{info['entity_id']} .\n\n"

                rdf += f"data:Candidate_{info['entity_id']} rdf:type onner:CandidateStatus ;\n"
                rdf += f"onner:statusAssignmentDate '{info['datetime']}'{xsd_datetime} ;\n"
                rdf += f"onner:statusAssignedBy data:{model} ;\n"

                label = info['label']
                label_number = schema_labels[label]
                
                rdf += f"onner:hasLabeledTermLabel data:{schema_name}_Label{label_number} .\n\n"

                # store all unique labels found in a file
                if label not in labels_found:
                    labels_found.update({label: label_number})
                        
        else:
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm data:NoLabeledTerm .\n\n"

    for key, value in labels_found.items():
        rdf += f"data:{schema_name}_Label{value} rdf:type onner:Label ;\n"
        rdf += f"onner:fromLabelingSchema data:Labeling_Schema ;\n"
        rdf += f"onner:labelText '{key}'{xsd_string} .\n\n"

    rdf += f"data:{schema_name}_Labels rdf:type onner:LabelingSchema ;\n"
    rdf += f"onner:schemaName '{schema_name}'{xsd_string} .\n\n"

    version = model.split('_')[-1][1:]
    rdf += f"data:{model} rdf:type onner:NER_System ;\n"
    rdf += f"onner:systemVersion '{version}'{xsd_string} .\n\n"

    return rdf


# GraphDBClient

**Helper functions:** \
*`_get_prefixes()`* => `insert_graph()` \
*`_parse_paragraphs()`* => `retrieve_paragraphs()`

### Insert RDF

In [6]:
from SPARQLWrapper import SPARQLWrapper, POST

# DEFINING PREFIXES

def _get_prefixes():
    prefixes = [
        'PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>',
        'PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>',
        'PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>',
        'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>',
        'PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>',
        'PREFIX owl: <http://www.w3.org/2002/07/owl#>',
    ]

    return '\n'.join(prefixes) + '\n'


# DATA INSERTION INTO GRAPH DB
def insert_graph(repository, document_id, rdf, rdf_type):

    # repository_name = 'GetTrainData'
    # database_url = f'http://dev:7200/repositories/{repository_name}/statements'
    end_point = f'http://dev:7200/repositories/{repository}/statements'
    
    if rdf_type == 'document':
        named_graph = f'http://purl.org/spatialai/onner/data/{document_id}/document'
    elif rdf_type == 'entities':
        named_graph = f'http://purl.org/spatialai/onner/data/{document_id}/entities'

    # query => insert data using named graph
    query = f'''
        {_get_prefixes()}

        INSERT DATA {{
            GRAPH <{named_graph}> {{
                {rdf}
            }}
        }}
    '''

    # execute query
    sparql = SPARQLWrapper(end_point)
    sparql.setMethod(POST)
    sparql.setQuery(query)

    try:
        sparql.query()
        print(f'RDF ({rdf_type}) successfully inserted into - {named_graph}')
    except Exception as e:
        print(f'Error: {e}')


### Retrieve Paragraphs

In [7]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import json

# def get_paragraph(exported_data):
def _parse_paragraphs(data):
    paragraphs = []

    for paragraph in data['results']['bindings']:
        paragraph_id = paragraph['paragraphId']['value'].split('#')[1]
        paragraph_text = paragraph['paragraphText']['value']
        paragraphs.append([paragraph_id, paragraph_text, {'entities': []}])
        
    return paragraphs


# DATA RETRIEVAL FROM GRAPH DB
# def get_document_content(publication_id):
# def _get_paragraphs(repository, publication_id):
def retrieve_paragraphs(repository, publication_id):  
    # specify the repository
    end_point = f'http://dev:7200/repositories/{repository}'
    sparql = SPARQLWrapper(end_point)

    # query => retrieving data
    try:
        sparql.setQuery(f'''
            PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>

            SELECT ?paragraphId ?paragraphText 
            WHERE {{
                    data:Publication_{publication_id} rdf:type onner:ScholarlyPublication ;
                                   onner:containsDocumentPart ?paragraphId .

                    ?paragraphId rdf:type onner:Paragraph ;
                                 onner:paragraphText ?paragraphText .
            }}
        ''')
        # convert results to JSON
        sparql.setReturnFormat(JSON)
        data = sparql.query().convert()
    except Exception as e:
        print(f'Error querying the SPARQL endpoint: {e}')

    return _parse_paragraphs(data)


# Entity Recognition

**Helper function:** \
*`_create_doc_object()`* => `extract_entities()`

In [8]:
import json
import spacy
from datetime import datetime


# NLP MODEL LOADER
def _create_doc_object(paragraph): 
    if not hasattr(_create_doc_object, 'nlp'):
        try:
            model_name = '/home/umayer/Work/research/cellograph_model/output/model-best'
            _create_doc_object.nlp = spacy.load(f'{model_name}')
            print('Model loaded successfully.')
        except OSError:
            print('ERROR: Model Not Found!')

    doc = _create_doc_object.nlp(paragraph) 
    
    return doc


# NAMED ENTITIES GENERATOR
def extract_entities(paragraphs):

    all_entities = []
    
    for i in paragraphs:
        paragraph_id = i[0]
        paragraph_text = i[1]
        entities_in_paragraph = []
        entity_number = 1
        
        doc = _create_doc_object(paragraph_text)     
        
        for ent in doc.ents:
            datetime_ = str(datetime.now())[:-7]    # RECHECK THE APPROPRIATE PLACEMENT
            entity_id = f'{paragraph_id}_E{entity_number}'
            entity = ent.text
            label = ent.label_
            offset = ent.start_char
            length = ent.end_char - ent.start_char
            # entity_info = (entity_id, entity, label, offset, length, datetime_)
            entity_info = {
                'entity_id': entity_id,
                'entity_text': entity,
                'label': label,
                'offset': offset, 
                'length': length,
                'datetime': datetime_
            }
            entities_in_paragraph.append(entity_info)
            entity_number += 1

        # all_entities.append([paragraph_id, entities_in_paragraph])
        all_entities.append(
            {
                'paragraph_id': paragraph_id,
                'entities': entities_in_paragraph
            }
        )
        
        print(f'Entity processed for {paragraph_id}')
        
    return all_entities


# Main

In [9]:
import time

repository = 'cgtest'
model = 'Model_CelloGraph_v1.0'  # this is a fixed value for now. 
labeling_schema = {
    'name': 'CelloGraph',
    'labels': {
        'CHEM_ENT':         1, 
        'MAT_ENT_STRUCT':   2, 
        'MAT_ENT_UNSTRUCT': 3,
        'PROPERTY':         4,
        'END_USE':          5,
        'PROCESS':          6,
        'EQUIPMENT':        7,
        'MEASUREMENT':      8,
        'ABBREVIATION':     9        
    }
}

start_time = time.perf_counter()

document_id, document_rdf = generate_document_rdf('/home/umayer/Work/research/cg_pipeline_test/Azeredo_2016.xml')
insert_graph(repository, document_id, document_rdf, 'document')
paragraphs = retrieve_paragraphs(repository, document_id)
entities = extract_entities(paragraphs)
entity_rdf = generate_entity_rdf(entities, labeling_schema, model)
insert_graph(repository, document_id, entity_rdf, 'entities')

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f'Execution time: {execution_time:.4f} seconds')


RDF (document) successfully inserted into - http://purl.org/spatialai/onner/data/10.1016_j.indcrop.2016.03.013/document


/home/umayer/Work/setup/_venv/cellograph/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded successfully.
Entity processed for 10.1016_j.indcrop.2016.03.013_A_P1
Entity processed for 10.1016_j.indcrop.2016.03.013_S1_P1
Entity processed for 10.1016_j.indcrop.2016.03.013_S1_P2
Entity processed for 10.1016_j.indcrop.2016.03.013_S1_P3
Entity processed for 10.1016_j.indcrop.2016.03.013_S1_P4
Entity processed for 10.1016_j.indcrop.2016.03.013_S1_P5
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P1
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P2
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P3
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P4
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P5
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P6
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P7
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P8
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P9
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P10
Entity processed for 10.1016_j.indcrop.2016.03.013_S2_P11
Ent